# EDA по [датасету](https://www.kaggle.com/datasets/shivamb/netflix-shows) о проектах на платформе **Netflix**

## 1. Чтение данных из .csv файла

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

df = pd.read_csv("netflix_titles.csv")

df.head()
df.tail()

## 2. Первичный анализ данных

In [ ]:
df.shape    # размер таблицы
df.info()   # информация о типах данных в таблице
df.describe(include='all') # статистика о таблице

## 3. Проверка качества данных

### 3.1 Проверка пустых данных

In [ ]:
df.isnull().sum().sort_values(ascending=False)
sns.heatmap(df.isnull(), cbar=False)
plt.show()
df.dropna()

### 3.2 Проверка на дубликаты

In [ ]:
df.duplicated().sum()

### 3.3 Проверка категорий

In [ ]:
df['type'].value_counts()
df['rating'].value_counts()
df['country'].value_counts()
df['release_year'].value_counts()
df['director'].value_counts()

### 3.4 Проверка логистических ошибок

In [ ]:
df[df['release_year'] > 2026]
df[df['title'].isna()]

## 4. Обработка данных
### 4.1 Обработка даты

In [ ]:
if not pd.api.types.is_datetime64_any_dtype(df['date_added']):
    df['date_added'] = pd.to_datetime(df['date_added'].str.strip(), format='%B %d, %Y')

### 4.2 Обработка длительности

In [ ]:
df['duration_num'] = df['duration'].str.extract(r'(\d+)').astype(float)
df['duration_type'] = df['duration'].str.extract(r'([A-Za-z]+)')
df.head(10)

### 4.3 Обработка жанров

In [ ]:
df['genre_main'] = df['listed_in'].str.split(',').str[0]
df.head(10)

## 4.4 Обработка стран

In [ ]:
df['country_main'] = df['country'].str.split(',').str[0]
df.head(10)

## 4.5 Возраст фильма на момент добавления

In [ ]:
df['content-age'] = df['date_added'].dt.year - df['release_year']
df.head(10)

## 5. Работа с пропусками

In [ ]:
df['country'] = df['country'].fillna('Unknown')
df['director'] = df['director'].fillna('Unknown')
df['cast'] = df['cast'].fillna('Unknown')
df.head(10)

## 6. Основные визуализации
### 6.1 Распределение типов контента

In [ ]:

px.histogram(df, x='type', title="Distribution of Content Type")

### 6.2 Контент по годам

In [ ]:
sns.histplot(data=df['release_year'], bins=30)
plt.title('Content release date distribution')
plt.show()

### 6.3 Топ стран

In [ ]:
top_countries = df['country_main'].value_counts().head(10)
px.bar(x=top_countries.values, y=top_countries.index, title='Countries Top')

### 6.4 Топ жанров

In [ ]:
top_genres = df['genre_main'].value_counts().head(10)
px.bar(top_genres.values, x=top_genres.values, y=top_genres.index, title='Genres Top')

### 6.5 Продолжительность фильмов

In [ ]:
movies = df[df['type'] == 'Movie']
px.histogram(movies, x='duration_num', title='Movies Duration')

### 6.6 Возраст контента на момент добавления в Netflix

In [ ]:
plt.hist(df['content-age'], bins=30)
plt.title('Distribution of content age')
plt.xlabel('Years between release and Netflix addition')
plt.ylabel('Count')
plt.show()

### 6.7 Количество сезонов в сериалах

In [ ]:
series = df[df['type'] == 'TV Show']
px.histogram(series, x='duration_num', title='Seasons amount')

## 6.8 Дата выхода на платформу

In [ ]:
px.histogram(df['date_added'], x=df['date_added'].dt.year)

## 7. Расширенная статистика
### 7.1 Статистика по длительности фильмов

In [ ]:
movies['duration_num'].min()
movies['duration_num'].max()
movies['duration_num'].mode()
movies['duration_num'].median()
movies['duration_num'].mean()
movies['duration_num'].quantile([0.25, 0.5, 0.75])
movies['duration_num'].var()
movies['duration_num'].skew()
movies['duration_num'].kurt()

## 8. Feature Engineering
### 8.1 OneHot Encoding

In [ ]:
df['type'].head()
df_encoded = pd.get_dummies(df, columns=['type', 'rating'], drop_first=False)
df_encoded['type_TV Show']

In [ ]:
df

## **Итоговые выводы**
### *Что я понял про датасет* :
- В датасете фильмов больше, чем TV Show
- Основной производитель фильмов - США
- Сейчас Нетфликс мгновенно добавляет контент на площадку
- Большинство фильмов имеет продолжительность 90-120 минут
- Драмы и комедии имеют преимущество среди всех жанров
- Большинство сериалов имеют 1-2 сезона
- Большая часть контента была добавлена в последние годы, что говорит о развитии платформы
### Дальнейшие действия
Дальше я бы обучал модели, которые предсказывают основной жанр фильма, дату выхода, рейтинг
